# Fetch the 200 listening clips

What this does
- Finds the 200 sampled clips in the shared audio folder (through your shortcut in My Drive).
- Saves them as `L001.wav` to `L200.wav` in a new Drive folder, `Listening_Clips`.
- Checks each clip is 3 seconds long.
- Writes `Listening_Clips.tar` in My Drive. Download it from there.

Why neutral names: the original names show recorder and time. Neutral names keep you blind to the
YAMNet label and the clip position. The list in the second code cell is packed and hidden on purpose.
Do not open `sample_key.csv` until the review sheet is complete.

Run the cells from top to bottom. Colab asks you to sign in twice (Drive access).

In [ ]:
# 1. Sign in and mount Drive (Drive is used to save the clips; the API is used to find them)
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import base64, gzip, io, os, shutil, tarfile, wave
import pandas as pd
import logging
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)  # hides a harmless timeout warning

service = build('drive', 'v3', cache_discovery=False)
OUT_NAME = 'Listening_Clips'
print('Signed in.')

In [ ]:
#@title 2. Clip list (packed, please do not decode) { display-mode: "form" }
BLOB = "H4sIAAAAAAAC/31Zy44kRwi8+1tWFu/MPPpuf0Nr7VlpR1rv+uDH77t6ponsGhXcQ0BBEEDW799ev7/cXl8+/fHt9a/b989/fvnpVyL+9Ms/L68/br/9+PvrTW9C4qS8bjxoivz83+d/7yh5Rnmixh3FSonSZ1Q8UEIHysJXoqxCBZMA5c8oSxTf2I5oOFFRodSXwtbobI1Ezcu47rZ4siVqFSiazpxxMV16vGeCFwHFV7buWY2nfLFcoo4KhQjiYq3jmuqJsss6zsMWTYZHLytEQ5MTHHW+wjVRl7k/oqd5YGeiZlNHQ/SrymroosyEUI2aHIniEmWisHXJ+7c6KqHacso9P33jUMc3Ss17mgSU11kdjuij5NcRF1Cn3Mtzb7MJPM6ShTq3x1XGJQQWKpVcZdsobqK39KhScyIouaradK2kMqnVFVJUSP2yO+5cdUWFNOo6BmV36Cg7jbeu6izzpbxtrUqjw6dlpxnVmSCoiXEZlxB4b1J07dEdbrClVXeEDOTLWr1H9F7aOr4y62jR2EK+bNQV8m2r4z3mkK1Svywsv9Gvef+mq+QZl3PHnFQ5b3KvBltaewwouVvnEdF7PbfVkhMetZJPyjr6qFm4kHuftUffHlczh4AKqmdagKtR854ZEznkUjHnfc9xTJjQJqtgTlg9txVaGI3er/2NUW5p9y0gUR3vsQPEbKLHTItVZ2J7HNTtTLlZDa62x6ODwOhR5j7cLZkztOwO1x2X1cq0KzS8U4Bk9IhOTXJajVHuX7aQ1dHumECturcn5uOseX+fC4niYvLddwBPfk1ppjum6NSC0bQOTiS/pnX7RGZ1ejmRzXBRzCgm8mFLscHMUc9tQQ/NdseErVV/o0MnFnXTPW2tTu+xRy8ptrSDnwt1XNrthZmvZbUtQncs7/QLtqL5RujqavR+oNPWbCq0o280x5P3x5xvPGa1mZrce34jU6P3lN/IpA1XBR4b3g8biepuK8c3Rn2dLIXHRu9X6hdTo/dz56ubtZJxcbPneG6iXNy179tQKjmzNHda7kzc3bWSisnc7DmDYMvrCbN29M2bguebAnOrOYhr1nNIdSaquWsp1YSFummVFZLLWfsW/UFVT5R0XM3oRZuXh+3Rmn5MlWPpeI86SnR3Bzx2mrNtzc5j9uP5rj3NRyFKj9rtOcjE9V37iD7Vl7W7rdDb2uY+e0ib3G8F0Cb3uHS4u2sXuKpd7qFyOut3k+GIq9Ecy3uIz3ftuUKmaet81557W/Ie4vNdyx/26GTO9V37PvkG+tGaHZMV0Xt3/WYdrX3HTH51d+1Ahbq7ds9HWw2jYctr3j9xortrFzLhUr5GHdd/dsf5rj1PZM39i93qbRv3I7t3XZvM8SgVUydy791tBd6f79oPL4Hg1/muPWs05WsUR6P3eyIHd3M7+RXSbHz5NsShHQszX91dK1CT8OZ+ZNiKerovbEPXd+3DI3Ifs9FVR1ZXjWJUaNT7vRhYeH3XviuAo0JDGgXAN45uv0cPjXq/t8AGM7zsNKGNimpLO9acjRrd23Z27fmu/fDSrMjXql99NqPPd+2ZOYbtcXLtUaDRs8s9Jsxs3jEVFZrtjgmP3r3BAFW/5xzdgW/s3tLA+zlrLdyzdq76lcywWS1q/pRRxlXctY+4stqr/ndiOxNLm/9D6Mdl3badmVhe/6UkB6rWnINeiGuU7wDHV2ZW1+z+isDjqrehyOiluGsfE+Z9pv0Pdpla9MMdAAA="
clips = pd.read_csv(io.StringIO(gzip.decompress(base64.b64decode(BLOB)).decode('utf-8')))
assert len(clips) == 200 and clips['blind_id'].is_unique
print(len(clips), 'clips to fetch')

In [ ]:
# 3. Find the clips by exact name, and accept only a copy at
#    Audio Moth Data / Segmented Foldered / Audio Moth N / <range folder>, with N the clip's own recorder.
#    Copies elsewhere (Segmented, team Toucanet Data, ...) are ignored.
import re

def list_by_name(names):
    q = '(' + ' or '.join(f"name = '{n}'" for n in names) + ') and trashed = false'
    found, token = [], None
    while True:
        r = service.files().list(q=q, corpora='user', supportsAllDrives=True, includeItemsFromAllDrives=True,
                                 fields='nextPageToken, files(id, name, size, md5Checksum, modifiedTime, parents, mimeType)',
                                 pageSize=200, pageToken=token).execute(num_retries=5)
        found += r.get('files', [])
        token = r.get('nextPageToken')
        if not token:
            return found

name_cache = {}
def folder_path(file_id, depth=0):
    if depth > 8:
        return '...'
    if file_id not in name_cache:
        try:
            name_cache[file_id] = service.files().get(fileId=file_id, fields='name, parents', supportsAllDrives=True).execute(num_retries=3)
        except Exception:
            name_cache[file_id] = {'name': '?', 'parents': []}
    info = name_cache[file_id]
    up = folder_path(info['parents'][0], depth + 1) + ' / ' if info.get('parents') else ''
    return up + info['name']

PATH_RE = re.compile(r'Audio Moth Data / Segmented Foldered / Audio Moth (\d) / \d+_\d+$')
def valid_source(f, clip_name):
    if not f.get('parents'):
        return False
    m = PATH_RE.search(folder_path(f['parents'][0]))
    return bool(m) and clip_name.startswith(f'Audio_Moth_{m.group(1)}_')

names = list(clips['clip_name'])
by_name = {}
for i in range(0, len(names), 20):
    for f in list_by_name(names[i:i + 20]):
        if f.get('mimeType') == 'application/vnd.google-apps.shortcut':
            continue  # a shortcut points at a real file that is listed on its own
        by_name.setdefault(f['name'], []).append(f)

chosen, missing, ambiguous, ignored = {}, [], [], 0
for n in names:
    all_copies = by_name.get(n, [])
    options = [f for f in all_copies if valid_source(f, n)]
    ignored += len(all_copies) - len(options)
    if not options:
        missing.append(n)
        continue
    if len(options) > 1:
        ambiguous.append(n)
    chosen[n] = options[0]
    by_name[n] = options

print(f'Found {len(chosen)} of {len(names)} in Segmented Foldered. Missing: {len(missing)}.')
print(f'Copies ignored because they are outside Segmented Foldered / Audio Moth N: {ignored}')
if missing:
    print('Missing (first 5):', missing[:5])

# Only if a name has two copies INSIDE Segmented Foldered: they must be the same file.
differing = [n for n in ambiguous if len({f.get('md5Checksum') for f in by_name[n]}) > 1]
print(f'Names with two copies inside Segmented Foldered: {len(ambiguous)}. Of those, not identical: {len(differing)}')
for n in differing:
    print(n)
    for f in by_name[n]:
        print('   ', f.get('size'), 'bytes |', (f.get('md5Checksum') or 'no checksum')[:8], '|', f.get('modifiedTime', '')[:10], '|', folder_path(f['parents'][0]))

In [ ]:
# 4. Download each clip and save it under its neutral name in a new Drive folder
assert not missing, 'Some clips were not found. Fix access to the audio folder, then rerun cells 3 and 4.'
dest = f'/content/drive/MyDrive/{OUT_NAME}'
os.makedirs(dest, exist_ok=True)
local = '/content/clips'
os.makedirs(local, exist_ok=True)

failed = []
for row in clips.itertuples():
    meta = chosen[row.clip_name]
    path = f'{local}/{row.blind_id}.wav'
    try:
        request = service.files().get_media(fileId=meta['id'], supportsAllDrives=True)
        with open(path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
            done = False
            while not done:
                _, done = downloader.next_chunk(num_retries=5)
        shutil.copy2(path, f'{dest}/{row.blind_id}.wav')
    except Exception as error:
        failed.append((row.blind_id, type(error).__name__))
print(f'Saved {len(clips) - len(failed)} of {len(clips)} clips to My Drive/{OUT_NAME}')
if failed:
    print('Failed (first 5):', failed[:5], '. If the error is about downloading, the owner has blocked downloads for this folder.')

In [ ]:
# 5. Check every clip is 3 seconds long and not empty, then write the tar
bad = []
for row in clips.itertuples():
    p = f'/content/clips/{row.blind_id}.wav'
    try:
        with wave.open(p, 'rb') as w:
            seconds = w.getnframes() / w.getframerate()
        if abs(seconds - 3.0) > 0.05:
            bad.append((row.blind_id, round(seconds, 2)))
    except Exception as error:
        bad.append((row.blind_id, type(error).__name__))
print('Clips that are not 3 seconds or cannot be read:', bad if bad else 'none')

tar_path = '/content/drive/MyDrive/Listening_Clips.tar'
with tarfile.open(tar_path, 'w') as tar:
    for row in clips.itertuples():
        tar.add(f'/content/clips/{row.blind_id}.wav', arcname=f'Listening_Clips/{row.blind_id}.wav')
print('Wrote', tar_path, round(os.path.getsize(tar_path) / 1e6, 1), 'MB')

In [ ]:
# 6. Optional: download the tar straight to your computer (it is also in My Drive)
from google.colab import files
files.download('/content/drive/MyDrive/Listening_Clips.tar')